## Import

In [1]:
import os
import pyspark
from pyspark.sql import SparkSession, functions as F

## Spark Session

In [2]:
spark = SparkSession.builder \
                    .master('local[*]') \
                    .appName('SparkSQL') \
                    .getOrCreate()

In [4]:
spark.version

'4.1.1'

In [5]:
spark.stop()

## Script

In [4]:
dfGreen = spark.read.parquet('data/pq/green/*/*')

In [6]:
dfYellow = spark.read.parquet('data/pq/yellow/*/*')

In [7]:
#check intersection of columns
set(dfGreen.columns) & set(dfYellow.columns)

{'DOLocationID',
 'PULocationID',
 'RatecodeID',
 'VendorID',
 'congestion_surcharge',
 'extra',
 'fare_amount',
 'improvement_surcharge',
 'mta_tax',
 'passenger_count',
 'payment_type',
 'store_and_fwd_flag',
 'tip_amount',
 'tolls_amount',
 'total_amount',
 'trip_distance'}

In [8]:
dfGreen = dfGreen \
            .withColumnRenamed('lpep_pickup_datetime','pickup_datetime') \
            .withColumnRenamed('lpep_dropoff_datetime','dropoff_datetime')

In [9]:
dfGreen = dfGreen \
            .withColumnRenamed('tpep_pickup_datetime','pickup_datetime') \
            .withColumnRenamed('tpep_dropoff_datetime','dropoff_datetime')

In [11]:
commonColumns =[]

yellowColumns = set(dfYellow.columns)

for col in dfGreen.columns:
    if col in yellowColumns:
        commonColumns.append(col)

In [12]:
dfGreen = dfGreen \
    .select(commonColumns) \
    .withColumn('service_type', F.lit('green'))

In [13]:
dfYellow = dfYellow \
    .select(commonColumns) \
    .withColumn('service_type', F.lit('yellow'))

In [14]:
dfTripsData = dfGreen.unionAll(dfYellow)

In [15]:
dfTripsData.registerTempTable('trips_data')

c:\Users\diogo.martins\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyspark\sql\classic\dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [16]:
spark.sql("""
SELECT
    COUNT(*)
FROM trips_data
"""    
).show()

+--------+
|count(1)|
+--------+
|41953716|
+--------+

